In [1]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

try:
    import irksome
except ImportError:
    !python3 -m pip install --no-dependencies git+https://github.com/firedrakeproject/Irksome.git#egg=Irksome
    import irksome

Solving monodomain equations with Fitzhugh-Nagumo reaction and a DIRK-IMEX method
=================================================================================

We're solving monodomain (reaction-diffusion) with a particular reaction term.
The basic form of the equation is:

$$
\chi \left( C_m u_t + I_{ion}(u) \right) = \nabla \cdot \sigma \nabla u
$$

where $u$ is the membrane potential, $\sigma$ is the conductivity tensor, $C_m$ is the specific capacitance of the cell membrane, and $\chi$ is the surface area to volume ratio.  The term $I_{ion}$ is current due to ionic flows through channels in the cell membranes, and may couple to a complicated reaction network.  In our case, we take the relatively simple model due to Fitzhugh and Nagumo.  Here, we have a separate concentration $c$ satisfying the reaction equation:

$$
   c_t = \epsilon( u + \beta - \gamma c)
$$

for certain positive parameters $\beta$ and $\gamma$, and the current takes the form of:

$$
   I_{ion}(u, c) = \tfrac{1}{\epsilon} \left( u - \tfrac{u^3}{3} - c \right)
$$

so that we have an overall system of two equations.  One of them is linear but stiff/diffusive, and the other is nonstiff but nonlinear.  This combination makes the system a good candidate for IMEX-type methods.  In this demo, we use some methods of Ascher, Ruuth, and Spiteri (Applied Numerical Mathematics 1997)

We start with Firedrake/Irksome imports:

In [2]:
import copy

from firedrake import (And, Constant, VTKFile, Function, FunctionSpace,
                       RectangleMesh, SpatialCoordinate, TestFunctions,
                       as_matrix, conditional, dx, grad, inner, split)
from irksome import Dt, MeshConstant, TimeStepper, ARS_DIRK_IMEX
from irksome.labeling import explicit


Here, we set up the mesh and function space:

In [3]:
mesh = RectangleMesh(20, 20, 70, 70, quadrilateral=True)

V = FunctionSpace(mesh, "CG", 2)
Z = V * V

x, y = SpatialCoordinate(mesh)
MC = MeshConstant(mesh)
dt = MC.Constant(0.05)
t = MC.Constant(0.0)

And specify the physical constants and initial states:

In [4]:
eps = Constant(0.1)
beta = Constant(1.0)
gamma = Constant(0.5)

chi = Constant(1.0)
capacitance = Constant(1.0)

sigma1 = sigma2 = 1.0
sigma = as_matrix([[sigma1, 0.0], [0.0, sigma2]])


initial_potential = conditional(x < 3.5, Constant(2.0), Constant(-1.28791))
initial_cell = conditional(And(And(31 <= x, x < 39), And(0 <= y, y < 35)),
                          Constant(2.0), Constant(-0.5758))

uu = Function(Z)
vu, vc = TestFunctions(Z)
uu.sub(0).interpolate(initial_potential)
uu.sub(1).interpolate(initial_cell)

(u, c) = split(uu)


Now, we select the 4-stage numerical scheme of Ascher, Ruuth, and Spiteri:

In [5]:
butcher_tableau = ARS_DIRK_IMEX(4, 4, 3)
ns = butcher_tableau.num_stages

Now, write down the Irksome semidiscrete form using a label to tag the part that we handle explicitly.

In [6]:
F1 = (inner(chi * capacitance * Dt(u), vu)*dx
      + inner(grad(u), sigma * grad(vu))*dx
      + inner(Dt(c), vc)*dx - inner(eps * u, vc)*dx
      - inner(beta * eps, vc)*dx + inner(gamma * eps * c, vc)*dx)

F2 = inner((chi/eps) * (-u + (u**3 / 3) + c), vu)*dx

F = F1 + explicit(F2)


Solver parameters.  In a DIRK method, we solve for each stage separately, Here, we will use a field split scheme that hits the potential (diffusive) block with AMG and the reaction/mass matrix block with incomplete Cholesky:

In [7]:
  
params = {"snes_type": "ksponly",
          "ksp_monitor": None,
          "mat_type": "aij",
          "ksp_type": "fgmres",
          "pc_type": "fieldsplit",
          "pc_fieldsplit_type": "additive",
          "fieldsplit_0": {
              "ksp_type": "preonly",
              "pc_type": "gamg",
          },
          "fieldsplit_1": {
              "ksp_type": "preonly",
              "pc_type": "icc",
          }}



The DIRK-IMEX scheme also requires us to solve mass matrices, which is not hard:

In [8]:
mass_params = {"snes_type": "ksponly",
               "ksp_rtol": 1.e-8,
               "ksp_monitor": None,
               "mat_type": "aij",
               "ksp_type": "cg",
               "pc_type": "icc",
              }

Finally, we can pass these into the stepper and solve for the system.  Note the use of the keyword argument!

In [9]:
stepper = TimeStepper(F, butcher_tableau, t, dt, uu,
                      stage_type="dirkimex",
                      solver_parameters=params,
                      mass_parameters=mass_params)
for j in range(100):
    print(f"{float(t)}")
    stepper.advance()
    t.assign(float(t) + float(dt))


0.0
    Residual norms for firedrake_1_ solve.
    0 KSP Residual norm 2.832980187317e+02
    1 KSP Residual norm 4.627292237508e+00
    2 KSP Residual norm 3.860687972121e-01
    3 KSP Residual norm 5.525599197573e-16
    Residual norms for firedrake_0_ solve.
    0 KSP Residual norm 4.255010700523e+01
    1 KSP Residual norm 3.765066481167e+00
    2 KSP Residual norm 3.971855457788e-01
    3 KSP Residual norm 2.259759093285e-02
    4 KSP Residual norm 3.115527320103e-03
    5 KSP Residual norm 2.061259950374e-04
    Residual norms for firedrake_1_ solve.
    0 KSP Residual norm 1.317713212469e+02
    1 KSP Residual norm 3.315640139698e+00
    2 KSP Residual norm 2.682268693774e-01
    3 KSP Residual norm 2.336501892308e-16
    Residual norms for firedrake_0_ solve.
    0 KSP Residual norm 2.649510281102e+00
    1 KSP Residual norm 3.080999716988e-01
    2 KSP Residual norm 2.425552530388e-02
    3 KSP Residual norm 1.568289091675e-03
    4 KSP Residual norm 1.892310325223e-04
    5 K

And a picture of the solution:

In [10]:
%config InlineBackend.figure_format = 'svg'

import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (11, 6)

In [11]:
from firedrake.pyplot import tripcolor, triplot

fig, axes = plt.subplots()
collection = tripcolor(uu.subfunctions[0], axes=axes, cmap='coolwarm')
fig.colorbar(collection);